In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter, gaussian_filter
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
from skimage import io, measure, morphology, color, util

INPUT_DIR = Path("images")
OUTPUT_DIR = Path("multiscale_gmm_results")
SCALES = (3, 5, 9)              # odd neighborhood widths, in pixels
SCALE_WEIGHTS = (1/3, 1/3, 1/3)
N_COMPONENTS = 3
SIGNAL_COMPONENTS = 1           # 1=brightest component; 2=intermediate+bright
SIGMA_SMOOTH = 0.7
MIN_AREA = 3
MAX_FIT_PIXELS = 100_000
MAX_ITER = 40
TOL = 1e-5
SPATIAL_STRENGTH = 1.0          # 0 = ordinary intensity GMM
PROBABILITY_THRESHOLD = 0.5
RANDOM_STATE = 42

for sub in ("figures", "masks", "labels", "measurements"):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

paths = sorted(p for p in INPUT_DIR.iterdir()
               if p.is_file() and p.suffix.lower() in (".tif", ".tiff")) if INPUT_DIR.exists() else []
print(f"Found {len(paths)} TIFF files:", *[p.name for p in paths], sep="\n - ")
if len(paths) != 4:
    warnings.warn(f"Expected 4 test images, found {len(paths)}. The notebook will process all found TIFFs.")

Found 4 TIFF files:
 - img1.tif
 - img2.tif
 - img3.tif
 - img4.tif


In [2]:
def load_2d_image(path):
    raw = np.asarray(io.imread(path))
    if raw.ndim != 2:
        raise ValueError(f"{path.name}: expected 2D grayscale, got shape {raw.shape}")
    raw = raw.astype(np.float64)
    if not np.isfinite(raw).all():
        raise ValueError(f"{path.name}: non-finite pixel values")
    lo, hi = np.percentile(raw, [0, 99.9])
    if hi <= lo:
        raise ValueError(f"{path.name}: image is constant or has insufficient intensity range")
    normalized = np.clip((raw - lo) / (hi - lo), 0, 1)
    return raw, normalized


def component_logpdf(x, means, stds):
    # Returns array [K, H, W]
    x = x[None, :, :]
    m = means[:, None, None]
    s = np.maximum(stds[:, None, None], 1e-3)
    return -0.5 * ((x - m) / s)**2 - np.log(s) - 0.5*np.log(2*np.pi)

In [3]:
def multiscale_spatial_gmm(
    image, n_components=3, scales=(3, 5, 9), scale_weights=None,
    spatial_strength=1.0, max_iter=40, tol=1e-5,
    max_fit_pixels=100_000, random_state=42
):
    image = np.asarray(image, dtype=np.float64)
    if image.ndim != 2:
        raise ValueError("Expected a 2D image")
    if not scales or any(int(s) != s or s < 1 or s % 2 == 0 for s in scales):
        raise ValueError("scales must be positive odd integers")
    if not (0 <= spatial_strength <= 1):
        raise ValueError("spatial_strength must be between 0 and 1")

    weights = np.ones(len(scales)) if scale_weights is None else np.asarray(scale_weights, float)
    if len(weights) != len(scales) or np.any(weights < 0) or weights.sum() <= 0:
        raise ValueError("Invalid scale weights")
    weights = weights / weights.sum()

    rng = np.random.default_rng(random_state)
    pixels = image.ravel()
    fit_pixels = pixels
    if len(pixels) > max_fit_pixels:
        fit_pixels = rng.choice(pixels, size=max_fit_pixels, replace=False)

    initial = GaussianMixture(n_components=n_components, n_init=5,
                              random_state=random_state, reg_covar=1e-5,
                              max_iter=200).fit(fit_pixels.reshape(-1, 1))
    order = np.argsort(initial.means_.ravel())
    means = initial.means_.ravel()[order].copy()
    stds = np.sqrt(initial.covariances_.reshape(-1))[order].copy()
    mixing = initial.weights_[order].copy()

    history = []
    for iteration in range(max_iter):
        log_likelihood = component_logpdf(image, means, stds)

        # Component similarity in each neighborhood; use float arrays, no hard labels
        affinities = np.zeros((n_components, *image.shape), dtype=float)
        for k in range(n_components):
            bandwidth = max(float(stds[k]), 0.025)
            pixel_similarity = np.exp(-0.5 * ((image - means[k]) / bandwidth)**2)
            for size, w in zip(scales, weights):
                affinities[k] += w * uniform_filter(
                    pixel_similarity, size=int(size), mode="reflect"
                )

        affinities = np.maximum(affinities, 1e-12)
        spatial_prior = affinities / affinities.sum(axis=0, keepdims=True)

        # Blend ordinary GMM prior and spatially varying prior
        prior = ((1 - spatial_strength) * mixing[:, None, None]
                 + spatial_strength * spatial_prior)
        log_joint = log_likelihood + np.log(np.maximum(prior, 1e-12))
        log_norm = logsumexp(log_joint, axis=0)
        posterior = np.exp(log_joint - log_norm[None, :, :])

        # Iterative weighted Gaussian updates (heuristic generalized EM,
        # not an exact maximum-likelihood optimizer for the spatial model)
        masses = posterior.sum(axis=(1, 2))
        new_means = np.sum(posterior * image[None, :, :], axis=(1, 2)) / np.maximum(masses, 1e-12)
        new_var = np.sum(
            posterior * (image[None, :, :] - new_means[:, None, None])**2,
            axis=(1, 2)
        ) / np.maximum(masses, 1e-12)
        new_stds = np.sqrt(np.maximum(new_var, 1e-5))
        new_mixing = masses / masses.sum()

        delta = float(np.max(np.abs(new_means - means)))
        history.append(delta)
        means, stds, mixing = new_means, new_stds, new_mixing
        if delta < tol:
            break

    # Recompute final posterior using FINAL parameters
    log_likelihood = component_logpdf(image, means, stds)
    affinities = np.zeros((n_components, *image.shape), dtype=float)
    for k in range(n_components):
        bandwidth = max(float(stds[k]), 0.025)
        similarity = np.exp(-0.5 * ((image - means[k]) / bandwidth)**2)
        for size, w in zip(scales, weights):
            affinities[k] += w * uniform_filter(similarity, size=int(size), mode="reflect")
    spatial_prior = affinities / np.maximum(affinities.sum(axis=0, keepdims=True), 1e-12)
    prior = (1 - spatial_strength) * mixing[:, None, None] + spatial_strength * spatial_prior
    joint = log_likelihood + np.log(np.maximum(prior, 1e-12))
    posterior = np.exp(joint - logsumexp(joint, axis=0)[None, :, :])

    return {
        "posterior": posterior, "classes": posterior.argmax(axis=0),
        "means": means, "stds": stds, "mixing": mixing,
        "history": history, "iterations": len(history)
    }

In [4]:
def ordinary_gmm(image, n_components=3, max_fit_pixels=100_000, random_state=42):
    pixels = image.ravel()
    rng = np.random.default_rng(random_state)
    sample = rng.choice(pixels, size=max_fit_pixels, replace=False) if len(pixels) > max_fit_pixels else pixels
    gmm = GaussianMixture(n_components=n_components, n_init=5,
                          reg_covar=1e-5, random_state=random_state).fit(sample.reshape(-1, 1))
    order = np.argsort(gmm.means_.ravel())
    posterior = gmm.predict_proba(pixels.reshape(-1, 1))[:, order].T.reshape(
        n_components, *image.shape)
    return {"posterior": posterior, "classes": posterior.argmax(axis=0),
            "means": gmm.means_.ravel()[order]}


def create_rois(posterior, means, signal_components=1, threshold=0.5, min_area=3):
    if not 1 <= signal_components < len(means) + 1:
        raise ValueError("Invalid signal_components")
    brightest_indices = np.argsort(means)[-signal_components:]
    signal_probability = posterior[brightest_indices].sum(axis=0)
    mask = signal_probability >= threshold
    mask = morphology.remove_small_objects(mask, min_size=min_area, connectivity=2)
    labels = measure.label(mask, connectivity=2)
    return mask, labels, signal_probability


def roi_measurements(labels, raw_image, image_name):
    props = measure.regionprops_table(
        labels, intensity_image=raw_image,
        properties=["label", "area", "centroid", "equivalent_diameter_area",
                    "perimeter", "perimeter_crofton", "eccentricity", "solidity",
                    "mean_intensity", "max_intensity", "min_intensity"]
    )
    df = pd.DataFrame(props)
    p = df["perimeter_crofton"].to_numpy(dtype=float)
    a = df["area"].to_numpy(dtype=float)
    df["circularity_crofton"] = np.divide(
        4*np.pi*a, p**2, out=np.full(len(df), np.nan),
        where=p > 0
    )
    df.insert(0, "image", image_name)
    return df


def draw_numbered_contours(ax, labels, roi_df):
    for region in measure.regionprops(labels):
        mask = labels == region.label
        for contour in measure.find_contours(np.pad(mask.astype(float), 1), 0.5):
            ax.plot(contour[:, 1] - 1, contour[:, 0] - 1,
                    color="red", linewidth=0.65)
    # Explicit DataFrame columns avoid namedtuple name mangling of centroid-0/1
    for _, row in roi_df.iterrows():
        ax.text(row["centroid-1"], row["centroid-0"], str(int(row["label"])),
                color="yellow", fontsize=6, ha="center", va="center",
                bbox={"facecolor":"black", "alpha":0.45, "pad":0.2, "edgecolor":"none"})

In [ ]:
all_rois = []
summary = []
for index, path in enumerate(paths, start=1):
    print(f"[{index}/{len(paths)}] {path.name}")
    raw, image = load_2d_image(path)
    smoothed = gaussian_filter(image, SIGMA_SMOOTH)

    baseline = ordinary_gmm(smoothed, N_COMPONENTS, MAX_FIT_PIXELS, RANDOM_STATE)
    spatial = multiscale_spatial_gmm(
        smoothed, n_components=N_COMPONENTS, scales=SCALES,
        scale_weights=SCALE_WEIGHTS, spatial_strength=SPATIAL_STRENGTH,
        max_iter=MAX_ITER, tol=TOL,
        max_fit_pixels=MAX_FIT_PIXELS, random_state=RANDOM_STATE
    )

    mask0, labels0, prob0 = create_rois(
        baseline["posterior"], baseline["means"], SIGNAL_COMPONENTS,
        PROBABILITY_THRESHOLD, MIN_AREA
    )
    mask, labels, signal_prob = create_rois(
        spatial["posterior"], spatial["means"], SIGNAL_COMPONENTS,
        PROBABILITY_THRESHOLD, MIN_AREA
    )

    name = f"{index:03d}_{path.stem}"
    roi_df = roi_measurements(labels, raw, name)
    roi_df.to_csv(OUTPUT_DIR / "measurements" / f"{name}_ROIs.csv", index=False)
    all_rois.append(roi_df)

    io.imsave(OUTPUT_DIR / "masks" / f"{name}_mask.tif",
              (mask.astype(np.uint8) * 255), check_contrast=False)
    io.imsave(OUTPUT_DIR / "labels" / f"{name}_labels.tif",
              labels.astype(np.int32), check_contrast=False)

    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    axes[0, 0].imshow(image, cmap="gray")
    axes[0, 0].set_title("Input (normalized for display)")
    axes[0, 1].imshow(image, cmap="gray")
    axes[0, 1].contour(mask0, levels=[0.5], colors="red", linewidths=0.6)
    axes[0, 1].set_title(f"Standard GMM — {labels0.max()} ROIs")
    axes[0, 2].imshow(image, cmap="gray")
    axes[0, 2].contour(mask, levels=[0.5], colors="red", linewidths=0.6)
    axes[0, 2].set_title(f"Multi-scale Spatial GMM — {labels.max()} ROIs")
    im = axes[1, 0].imshow(signal_prob, vmin=0, vmax=1, cmap="viridis")
    axes[1, 0].set_title("Signal posterior probability")
    fig.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)
    axes[1, 1].imshow(mask, cmap="gray")
    axes[1, 1].set_title("Binary mask")
    axes[1, 2].imshow(image, cmap="gray")
    draw_numbered_contours(axes[1, 2], labels, roi_df)
    axes[1, 2].set_title("Numbered ROIs (CSV label IDs)")
    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle(path.name, fontsize=12)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures" / f"{name}_comparison.png", dpi=180)
    plt.show()
    plt.close(fig)

    summary.append({
        "image": name, "source_file": path.name,
        "standard_gmm_n_rois": int(labels0.max()),
        "multiscale_gmm_n_rois": int(labels.max()),
        "multiscale_total_area_px": int(mask.sum()),
        "iterations": spatial["iterations"],
        "component_means": ";".join(f"{v:.5f}" for v in spatial["means"])
    })

summary_df = pd.DataFrame(summary)
all_rois_df = pd.concat(all_rois, ignore_index=True) if all_rois else pd.DataFrame()
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)
all_rois_df.to_csv(OUTPUT_DIR / "all_ROIs.csv", index=False)
print(f"Saved to {OUTPUT_DIR.resolve()}")
display(summary_df)